# 练习：使用图像相减法进行图像对比

In [ ]:
import numpy as np              # NumPy数值计算库
import matplotlib.pyplot as plt  # Matplotlib绑图库
import cv2 as cv                 # OpenCV计算机视觉库

def show(img):
    """自定义显示函数：自动判断灰度图或彩色图并正确显示"""
    if img.ndim == 2:  # 灰度图（二维数组）
        plt.imshow(img, cmap='gray')
    else:  # 彩色图（三维数组），BGR转RGB
        plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    plt.show()

In [ ]:
# 读取两张待对比的图片
img_a = cv.imread('pic/2.13(a).png')
img_b = cv.imread('pic/2.13(b).png')
show(np.hstack([img_a, img_b]))  # 并排显示原图

In [ ]:
# NumPy直接相减：逐元素减法，uint8下负值会取模溢出（如0-100=156）
# 这导致差异区域变白，非差异区域也可能出现异常值
img_sub = img_b - img_a
show(img_sub)

In [ ]:
# cv.subtract()：OpenCV饱和减法，负值截断为0（不会取模）
# 结果偏暗是因为差值较小的区域被截断为0（黑色），只有差异明显的区域有值
img_sub2 = cv.subtract(img_b, img_a)
show(img_sub2)

In [ ]:
# 对减法结果进行高斯模糊去噪，再转为灰度图
blur = cv.GaussianBlur(img_sub, (5, 5), 0)  # 高斯模糊，核大小5x5
# print(blur)
# blur = blur.astype("uint8")
blur = cv.cvtColor(blur, cv.COLOR_BGR2GRAY)  # 转换为灰度图

# cv.threshold() + THRESH_OTSU：自动计算最佳阈值进行二值化
# OTSU算法自动确定阈值，适合双峰分布的直方图
threshold_value, binary_image = cv.threshold(blur, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
show(binary_image)

为什么同样是两张图像相减，一张图像偏白，另一张偏黑？

In [ ]:
# 演示NumPy减法的取模溢出行为
# 0代表黑色，255代表白色
x = np.uint8([100])
y = np.uint8([156])
print(x - y)  # 100-156=-56，uint8取模：-56%256=200，结果偏白

In [ ]:
# 演示cv.subtract()的饱和运算行为
# 0代表黑色，255代表白色
x = np.uint8([100])
y = np.uint8([156])
print(cv.subtract(x, y))  # 100-156=-56，饱和运算截断为0，结果偏黑